<a href="https://colab.research.google.com/github/BrayFlo/Mineria-Datos/blob/main/FuncionesAvanzadaspt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [65]:
import pandas as pd

In [66]:
orders_a = pd.read_csv("part2_orders_a.csv")
orders_b = pd.read_csv("part2_orders_b.csv")

Une orders_a y orders_b en un solo DataFrame orders con índice reiniciado.

In [67]:
# Unir en un solo DataFrame 'orders'
# ignore_index=True crea un nuevo conteo de filas (0, 1, 2...) para que no se repitan
orders = pd.concat([orders_a, orders_b], ignore_index=True)
print("Paso 1: DataFrame unido con índice reiniciado")
print(orders.head())

Paso 1: DataFrame unido con índice reiniciado
   order_id  order_date  line_id customer_id product_id  qty channel
0      2001  2026-01-02        1        C001        P10    2     web
1      2002  2026-01-02        2        C002        P20    1   store
2      2003  2026-01-03        3        C003        P20    3     web
3      2004  2026-01-04        4        C001        P30    1     app
4      2005  2026-01-05        5        C004        P10    5     web


Ordena orders por order_date ascendente y, en caso de empate, por qty descendente.

In [68]:
# Ordenar: fecha ASC, cantidad DESC
orders = orders.sort_values(by=['order_date', 'qty'], ascending=[True, False])

print("\nPaso 2: DataFrame ordenado")
print(orders.head())


Paso 2: DataFrame ordenado
   order_id  order_date  line_id customer_id product_id  qty channel
0      2001  2026-01-02        1        C001        P10    2     web
1      2002  2026-01-02        2        C002        P20    1   store
2      2003  2026-01-03        3        C003        P20    3     web
3      2004  2026-01-04        4        C001        P30    1     app
4      2005  2026-01-05        5        C004        P10    5     web


Establece order_date como índice en orders y ordena por índice (cronológico).

In [69]:
# 1. Asegurar formato de fecha (opcional pero recomendado)
orders['order_date'] = pd.to_datetime(orders['order_date'])

# 2. Establecer como índice
orders = orders.set_index('order_date')

# 3. Ordenar por el nuevo índice (cronológico)
orders = orders.sort_index()

print("\nPaso 3: Fecha establecida como índice y ordenado")
print(orders.head())


Paso 3: Fecha establecida como índice y ordenado
            order_id  line_id customer_id product_id  qty channel
order_date                                                       
2026-01-02      2001        1        C001        P10    2     web
2026-01-02      2002        2        C002        P20    1   store
2026-01-03      2003        3        C003        P20    3     web
2026-01-04      2004        4        C001        P30    1     app
2026-01-05      2005        5        C004        P10    5     web


Merge (enriquecer órdenes con clientes):
Haz un merge para añadir a orders las columnas del cliente (customer_name, city, segment) usando customer_id (left join). Valida que sea many_to_one.

In [70]:
# Cargar las nuevas tablas
customers = pd.read_csv("part2_customers.csv")
products = pd.read_csv("part2_products.csv")

# Si tu DataFrame 'orders' tiene el índice en la fecha,
# a veces es más fácil resetearlo para el merge y luego volverlo a poner
orders = orders.reset_index()

In [71]:
# Merge 1: Órdenes + Clientes
# Usamos 'on' para la columna llave y 'how' para el tipo de unión
orders = pd.merge(
    orders,
    customers[['customer_id', 'customer_name', 'city', 'segment']],
    on='customer_id',
    how='left',
    validate='many_to_one'
)

print("Paso 1: Órdenes enriquecidas con datos del cliente")
print(orders.head())

Paso 1: Órdenes enriquecidas con datos del cliente
  order_date  order_id  line_id customer_id product_id  qty channel  \
0 2026-01-02      2001        1        C001        P10    2     web   
1 2026-01-02      2002        2        C002        P20    1   store   
2 2026-01-03      2003        3        C003        P20    3     web   
3 2026-01-04      2004        4        C001        P30    1     app   
4 2026-01-05      2005        5        C004        P10    5     web   

  customer_name  city    segment  
0           Ana  CDMX   Consumer  
1          Luis   GDL  Corporate  
2         Marta   MTY   Consumer  
3           Ana  CDMX   Consumer  
4         Sofia   QRO  Small Biz  


Merge (añadir producto y calcular total):
Haz un merge adicional con products para añadir product, category, unit_price y crea una columna line_total = qty * unit_price. Luego ordena por line_total descendente y muestra top 5.

In [74]:
# Merge 2: Órdenes + Productos
# Añadimos las columnas necesarias de la tabla de productos
orders = pd.merge(
    orders,
    products[['product_id', 'product', 'category', 'unit_price']],
    on='product_id',
    how='left'
)

# Calcular el total de la línea (Cantidad * Precio Unitario)
orders['line_total'] = orders['qty'] * orders['unit_price']

# Ordenar por el total de mayor a menor y mostrar el Top 5
top_5_ventas = orders.sort_values(by='line_total', ascending=False).head(5)

print("\nTop 5 Ventas más altas:")
print(top_5_ventas[['order_id', 'customer_name', 'product', 'line_total']])


Top 5 Ventas más altas:
   order_id customer_name       product  line_total
6      2007          Luis   Monitor 24"       320.0
5      2006         Diego  Office Chair       220.0
3      2004           Ana   Monitor 24"       160.0
2      2003         Marta      Keyboard       135.0
9      2010         Ramon      Keyboard        90.0
